# Accrual Operations Worklist — Silver Layer

**Purpose:** Generate the accrual worklist at **Company Code / PO / PO Item / G/L Account / Accounting Document** granularity.

**Logic (Multi-Data-Product Joins):**
1. **GR (Goods Receipt):** `AccountingDocumentClass = 'WE'` + `LedgerGroup = '0L'` from `raw_journalentry` header joined to `raw_operationalacctgdocitem` line items
2. **IR (Invoice Receipt):** `AccountingDocumentClass = 'RE'` + `LedgerGroup = '0L'` from same join
3. **Signed GR Amount:** `SUM(AmountInCompanyCodeCurrency × sign(DebitCreditCode))` WHERE doc class = 'WE'
4. **Signed IR Amount:** `SUM(AmountInCompanyCodeCurrency × sign(DebitCreditCode))` WHERE doc class = 'RE'
5. **Exposure:** Signed GR Amount − Signed IR Amount (only where GR > IR, i.e. GR still open / partial IR / invoice missing)
6. **Days Open:** System Date – Oldest open GR date
   - Filter: `AccountingDocumentClass = 'WE'` (GR documents only)
   - Condition: GR is still open (partial IR or invoice missing → Exposure > 0)
   - Identify: `MIN(PostingDate)` → oldest GR posting date
   - Compute: `Days_Open = Current_Date - MIN(PostingDate)`
7. **Action:** POST / REVIEW / OBSERVE / NO ACTION (deterministic rule-based)

**V2 Extension (Day 2) — Additive on top of V1:**
1. **Posting Period Alignment:** FiscalYear + FiscalPeriod from journalentry header. Cross-Period Flag = 'Y' when GR period < IR period.
2. **Quantity Variance:** Signed_GR_Quantity − Signed_IR_Quantity. Quantity Exposure = Variance × Net Order Price.
3. **Price/Billing Variance:** When qty matched but amount differs → Billing_Variance = Signed_GR_Amount − Signed_IR_Amount.
4. **Missing Invoice:** GR exists (WE) with no IR (RE) at all → Invoice_Missing_Flag = 'Y'. Excluded if IsFinallyInvoiced or IsCompletelyDelivered.
5. **Days Open V2:** Reference Date − MIN(PostingDate). Fallback: Reference Date − PurchaseOrderDate (when no GR yet).
6. **Payment Netting:** GR/IR Accrual = Exposure (V1). Open Payable = IR − Paid. Open Amount Against PO = Accrual + Open Payable. (Paid Amount = down payments only until clearing fields confirmed.)
7. **Worklist Category** (first match wins): Invoice Missing → Cross-Period GR-IR → Quantity Variance → Price Variance → Matched-Unpaid → Fully Matched

**Data Products & Authoritative Sources:**
| Field | Authoritative Table | Data Product | Fallback |
|-------|--------------------|--------------|---------|
| CompanyCode | raw_companycode | companycode | raw_journalentry |
| AccountingDocument, AccountingDocumentClass | raw_journalentry | journalentryheader | — |
| LedgerGroup = '0L' | raw_journalentry | journalentryheader | — |
| PostingDate (header) | raw_journalentry | journalentryheader | — |
| Amount in CC Currency | raw_operationalacctgdocitem | operationalacctgdocitem | — |
| Debit/Credit Indicator | raw_debitcreditcode (validation) | journalentryitemcodes | raw_operationalacctgdocitem |
| PurchasingDocument (PO ref) | raw_operationalacctgdocitem | operationalacctgdocitem | — |
| PurchasingDocumentItem | raw_operationalacctgdocitem | operationalacctgdocitem | — |
| GLAccount (line item) | raw_operationalacctgdocitem | operationalacctgdocitem | raw_purchaseorderaccountassignment |
| PurchaseOrderQty | raw_operationalacctgdocitem | operationalacctgdocitem | — |
| PurchaseOrder | raw_purchaseorder | purchaseorder | raw_operationalacctgdocitem |
| PurchaseOrderItem | raw_purchaseorderitem | purchaseorder | raw_operationalacctgdocitem |
| PO Quantity (OrderQuantity) | raw_purchaseorderitem | purchaseorder | raw_operationalacctgdocitem |
| G/L Account (authoritative) | raw_purchaseorderaccountassignment | purchaseorder | raw_operationalacctgdocitem |
| Supplier/Vendor | raw_supplierinvoice (InvoicingParty) | supplierinvoice | raw_purchaseorder |
| Currency / ExchangeRate (USD) | raw_companycode | companycode | — |

**ER Diagram (Entity Relationships):**
```
┌─────────────────────────────────┐         ┌─────────────────────────────────────┐
│  raw_journalentry               │         │  raw_operationalacctgdocitem        │
│  (journalentryheader)           │         │  (operationalacctgdocitem)          │
│─────────────────────────────────│         │─────────────────────────────────────│
│  PK: CompanyCode                │ 1     N │  PK: CompanyCode                    │
│  PK: AccountingDocument         │────────▶│  PK: AccountingDocument             │
│  PK: FiscalYear                 │         │  PK: FiscalYear                     │
│                                 │         │  PK: AccountingDocumentItem          │
│  AccountingDocumentClass (WE/RE)│         │                                     │
│  LedgerGroup (0L)               │         │  DebitCreditCode (S/H)              │
│  PostingDate                    │         │  GLAccount                          │
│  AccountingDocumentType         │         │  AmountInCompanyCodeCurrency         │
│  CompanyCodeCurrency            │         │  AmountInTransactionCurrency         │
│  ExchangeRate                   │         │  PurchasingDocument ─────────┐       │
│                                 │         │  PurchasingDocumentItem ─────┤       │
└─────────────────────────────────┘         │  PurchaseOrderQty            │       │
                                            │  Supplier                    │       │
                                            │  PostingKey                  │       │
                                            │  FinancialAccountType        │       │
                                            │  ProfitCenter                │       │
                                            │  DocumentItemText            │       │
                                            └────────────────────────┬─────┘       
                                                                     │             
                          ┌──────────────────────────────────────────┘             
                          │ FK: PurchasingDocument + PurchasingDocumentItem        
                          ▼                                                        
┌─────────────────────────────────┐         ┌─────────────────────────────────────┐
│  raw_purchaseorder              │         │  raw_purchaseorderitem              │
│  (purchaseorder)                │         │  (purchaseorder)                    │
│─────────────────────────────────│         │─────────────────────────────────────│
│  PK: PurchaseOrder              │ 1     N │  PK: PurchaseOrder                  │
│                                 │────────▶│  PK: PurchaseOrderItem              │
│  CompanyCode                    │         │                                     │
│  Supplier ★                     │         │  OrderQuantity ★                    │
│  DocumentCurrency               │         │  NetAmount                          │
│  PurchaseOrderDate              │         │  PurchaseOrderItemText              │
│  InvoicingParty                 │         │  CompanyCode                        │
│  PurchaseOrderType              │         │  IsCompletelyDelivered              │
└─────────────────────────────────┘         │  IsFinallyInvoiced                  │
                                            │  InvoiceIsExpected                  │
                                            └─────────────────────────────────────┘
                                                         │                         
                                                         │ 1:1                     
                                                         ▼                         
┌─────────────────────────────────┐         ┌─────────────────────────────────────┐
│  raw_supplierinvoice            │         │  raw_purchaseorderaccountassignment │
│  (supplierinvoice)              │         │  (purchaseorder)                    │
│─────────────────────────────────│         │─────────────────────────────────────│
│  PK: SupplierInvoice            │         │  PK: PurchaseOrder                  │
│  PK: FiscalYear                 │         │  PK: PurchaseOrderItem              │
│                                 │         │  PK: AccountAssignmentNumber        │
│  CompanyCode                    │         │                                     │
│  InvoicingParty ★ (Supplier)    │         │  GLAccount ★ (authoritative)        │
│  DocumentCurrency               │         │  CostCenter                         │
│  InvoiceGrossAmount             │         │  ProfitCenter                       │
│  PostingDate                    │         │  PurgDocNetAmount                   │
└─────────────────────────────────┘         └─────────────────────────────────────┘
                                                                                   
┌─────────────────────────────────┐         ┌─────────────────────────────────────┐
│  raw_companycode                │         │  raw_debitcreditcode                │
│  (companycode)                  │         │  (journalentryitemcodes)            │
│─────────────────────────────────│         │─────────────────────────────────────│
│  PK: CompanyCode                │         │  PK: DebitCreditCode                │
│                                 │         │                                     │
│  Currency (CC Currency) ★       │         │  S = Debit (Soll)                   │
│  CompanyCodeName                │         │  H = Credit (Haben)                 │
│  Country                        │         │                                     │
│  ChartOfAccounts                │         │  (validation / reference only)      │
└─────────────────────────────────┘         └─────────────────────────────────────┘

★ = Authoritative source for that field
PK = Primary Key
FK = Foreign Key
1:N = One-to-Many relationship

SUPPLIER RESOLUTION:  supplierinvoice.InvoicingParty → purchaseorder.Supplier → acctgdocitem.Supplier
GL ACCOUNT RESOLUTION: purchaseorderaccountassignment.GLAccount → acctgdocitem.GLAccount
```

**Join Structure:**
```
┌──────────────────────────────────────────────────────────────────────┐
│  raw_journalentry (HEADER)              [journalentryheader]         │
│  LedgerGroup='0L', AccountingDocumentClass IN ('WE','RE')            │
│  PostingDate, CompanyCode, AccountingDocument, FiscalYear            │
└─────────────────────────────┬────────────────────────────────────────┘
                              │ INNER JOIN ON
                              │ CompanyCode + AccountingDocument + FiscalYear
                              ▼
┌──────────────────────────────────────────────────────────────────────┐
│  raw_operationalacctgdocitem (LINE ITEMS) [operationalacctgdocitem]  │
│  DebitCreditCode, GLAccount, AmountInCompanyCodeCurrency             │
│  PurchasingDocument, PurchasingDocumentItem, PurchaseOrderQty        │
│  Supplier                                                            │
└─────────────────────────────┬────────────────────────────────────────┘
                              │ LEFT JOINs (enrichment)
              ┌───────────────┼───────────────────────┐
              ▼               ▼                       ▼
┌─────────────────────┐ ┌──────────────────┐ ┌────────────────────────┐
│ raw_supplierinvoice │ │ raw_purchaseorder │ │ raw_purchaseorderitem  │
│ [supplierinvoice]   │ │ [purchaseorder]   │ │ [purchaseorder]        │
│                     │ │                   │ │                        │
│ InvoicingParty ★    │ │ Supplier (fb)     │ │ OrderQuantity ★        │
│ (Supplier AUTH)     │ │ DocumentCurrency  │ │ NetAmount              │
│ JOIN ON CompanyCode │ │ JOIN ON PO        │ │ PurchaseOrderItemText  │
└─────────────────────┘ └──────────────────┘ │ JOIN ON PO + POItem    │
                                             └────────────────────────┘
              ┌───────────────────────────────────────┐
              ▼                                       ▼
┌──────────────────────────────┐  ┌────────────────────────────────────┐
│ raw_purchaseorderaccount-    │  │ raw_companycode [companycode]       │
│ assignment [purchaseorder]   │  │                                    │
│                              │  │ Currency (CompanyCodeCurrency)      │
│ GLAccount ★ (authoritative)  │  │ CompanyCodeName                    │
│ JOIN ON PO + POItem          │  │ JOIN ON CompanyCode                │
└──────────────────────────────┘  └────────────────────────────────────┘
              ┌───────────────────────────────────────┐
              ▼                                       │
┌──────────────────────────────┐                      │
│ raw_debitcreditcode          │  ★ = Authoritative   │
│ [journalentryitemcodes]      │  (fb) = Fallback     │
│                              │                      │
│ INNER JOIN = validation      │  SUPPLIER ORDER:     │
│ DebitCreditCode              │  1. supplierinvoice  │
└──────────────────────────────┘  2. purchaseorder    │
                                  3. acctgdocitem     │
                                                      │
                                  GL ACCOUNT ORDER:   │
                                  1. PO acct assign   │
                                  2. acctgdocitem     │
                                  └───────────────────┘
```

In [0]:
# ── Configuration ────────────────────────────────────────────────────────────
CATALOG = "[pe]_xx102_pe_finance"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
TARGET_TABLE = "accrual_worklist"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{SCHEMA_SILVER}`")
spark.sql(f"USE SCHEMA `{SCHEMA_SILVER}`")

print(f"Target: {CATALOG}.{SCHEMA_SILVER}.{TARGET_TABLE}")

Target: [pe]_xx102_pe_finance.silver.accrual_worklist


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ── Step 1: GR (Goods Receipt) — Multi-Data-Product Join ───────────────────
# Core: raw_journalentry HEADERS + raw_operationalacctgdocitem LINE ITEMS
# Join: CompanyCode + AccountingDocument + FiscalYear
# Enrichment:
#   raw_purchaseorder         → DocumentCurrency
#   raw_supplierinvoice       → InvoicingParty = Supplier (authoritative)
#   raw_purchaseorderitem     → OrderQuantity (authoritative PO Qty), PurchaseOrderItemText
#   raw_purchaseorderacctassignment → GLAccount (authoritative)
#   raw_companycode           → Currency (for USD standardization)
#   raw_debitcreditcode       → validate DebitCreditCode values
#   (Supplier fallback: raw_purchaseorder, raw_operationalacctgdocitem)

# ---- Load core tables: header + line items ----
je_header = spark.table(f"`{CATALOG}`.`{SCHEMA_BRONZE}`.raw_journalentry")
oa_items = spark.table(f"`{CATALOG}`.`{SCHEMA_BRONZE}`.raw_operationalacctgdocitem")

# Join header (has LedgerGroup, AccountingDocumentClass) with line items (has amounts, GL, PO ref)
je = (
    je_header
    .filter(
        (F.col("LedgerGroup") == "0L") &
        (F.col("AccountingDocumentClass").isin("WE", "RE"))
    )
    .select(
        F.col("CompanyCode").alias("JE_CompanyCode"),
        F.col("AccountingDocument").alias("JE_AccountingDocument"),
        F.col("FiscalYear").alias("JE_FiscalYear"),
        F.col("AccountingDocumentClass"),
        F.col("LedgerGroup"),
        F.col("PostingDate").alias("JE_PostingDate"),
        # V2: Period awareness
        F.col("FiscalPeriod").alias("JE_FiscalPeriod")
    )
    .join(
        oa_items,
        (F.col("JE_CompanyCode") == oa_items["CompanyCode"]) &
        (F.col("JE_AccountingDocument") == oa_items["AccountingDocument"]) &
        (F.col("JE_FiscalYear") == oa_items["FiscalYear"]),
        how="inner"
    )
    .withColumn("PostingDate", F.coalesce(oa_items["PostingDate"], F.col("JE_PostingDate")))
    .withColumn("FiscalPeriod", F.col("JE_FiscalPeriod"))
    .withColumn("FiscalYear", F.col("JE_FiscalYear"))
    .drop("JE_CompanyCode", "JE_AccountingDocument", "JE_FiscalYear", "JE_PostingDate", "JE_FiscalPeriod")
)

# Data Product: purchaseorder
po_header = (
    spark.table(f"`{CATALOG}`.`{SCHEMA_BRONZE}`.raw_purchaseorder")
    .select(
        F.col("PurchaseOrder").cast("double").alias("PO_Key"),
        F.col("Supplier").alias("PO_Supplier"),
        F.col("CompanyCode").alias("PO_CompanyCode"),
        F.col("DocumentCurrency"),
        # V2: PO Date for Days Open fallback
        F.col("PurchaseOrderDate").alias("PO_Date")
    )
)

po_item = (
    spark.table(f"`{CATALOG}`.`{SCHEMA_BRONZE}`.raw_purchaseorderitem")
    .select(
        F.col("PurchaseOrder").cast("double").alias("POI_Key"),
        F.col("PurchaseOrderItem").cast("long").alias("POI_Item"),
        F.col("OrderQuantity").alias("PO_OrderQuantity"),
        F.col("NetAmount").alias("PO_NetAmount"),
        F.col("PurchaseOrderItemText"),
        # V2: Net Order Price for Quantity Exposure valuation
        F.col("NetPriceAmount").alias("PO_NetPrice"),
        # V2: Completion indicators for Missing Invoice exclusion
        F.col("IsCompletelyDelivered").alias("PO_IsCompletelyDelivered"),
        F.col("IsFinallyInvoiced").alias("PO_IsFinallyInvoiced")
    )
)

po_acct = (
    spark.table(f"`{CATALOG}`.`{SCHEMA_BRONZE}`.raw_purchaseorderaccountassignment")
    .select(
        F.col("PurchaseOrder").cast("double").alias("POA_Key"),
        F.col("PurchaseOrderItem").cast("long").alias("POA_Item"),
        F.col("GLAccount").alias("POA_GLAccount")
    )
    .filter(F.col("POA_GLAccount").isNotNull())
)

# Data Product: companycode
company = (
    spark.table(f"`{CATALOG}`.`{SCHEMA_BRONZE}`.raw_companycode")
    .select(
        F.col("CompanyCode").alias("CC_Key"),
        F.col("Currency").alias("CC_Currency"),
        F.col("CompanyCodeName").alias("CC_CompanyCodeName")
    )
)

# Data Product: journalentryitemcodes (reference validation)
dcc = (
    spark.table(f"`{CATALOG}`.`{SCHEMA_BRONZE}`.raw_debitcreditcode")
    .select(F.col("DebitCreditCode").alias("DCC_Code"))
)

# Data Product: supplierinvoice (Supplier - authoritative)
# Join via supplierinvoiceitem to get the SPECIFIC supplier for each PO item
si_item = (
    spark.table(f"`{CATALOG}`.`{SCHEMA_BRONZE}`.raw_supplierinvoiceitem")
    .select(
        F.col("PurchaseOrder").cast("double").alias("SII_PO"),
        F.col("PurchaseOrderItem").cast("long").alias("SII_POItem"),
        F.col("SupplierInvoice").alias("SII_Invoice"),
        F.col("FiscalYear").alias("SII_FiscalYear")
    )
    .filter(F.col("SII_PO").isNotNull())
)

si_header = (
    spark.table(f"`{CATALOG}`.`{SCHEMA_BRONZE}`.raw_supplierinvoice")
    .select(
        F.col("SupplierInvoice").alias("SI_Invoice"),
        F.col("FiscalYear").alias("SI_FiscalYear"),
        F.col("InvoicingParty").alias("SI_Supplier")
    )
    .filter(F.col("SI_Supplier").isNotNull())
)

# Join item → header to get supplier per PO+POItem (no fan-out)
si_supplier = (
    si_item
    .join(
        si_header,
        (F.col("SII_Invoice") == F.col("SI_Invoice")) &
        (F.col("SII_FiscalYear") == F.col("SI_FiscalYear")),
        how="inner"
    )
    .select(
        F.col("SII_PO").alias("SI_PO"),
        F.col("SII_POItem").alias("SI_POItem"),
        F.col("SI_Supplier")
    )
    .dropDuplicates(["SI_PO", "SI_POItem"])
)

# ---- GR Core: Filter for Goods Receipts (WE) ----
gr_lines = (
    je
    .filter(
        (F.col("AccountingDocumentClass") == "WE") &
        (F.col("PurchasingDocument").isNotNull()) &
        (F.col("DebitCreditCode").isin("S", "H"))
    )
    # Validate DebitCreditCode against reference table (journalentryitemcodes)
    .join(dcc, F.col("DebitCreditCode") == F.col("DCC_Code"), how="inner")
    .drop("DCC_Code")
    # Enrich with purchaseorder data product
    .join(po_header, F.col("PurchasingDocument") == F.col("PO_Key"), how="left")
    .join(po_item, (F.col("PurchasingDocument") == F.col("POI_Key")) & 
                   (F.col("PurchasingDocumentItem") == F.col("POI_Item")), how="left")
    .join(po_acct, (F.col("PurchasingDocument") == F.col("POA_Key")) & 
                   (F.col("PurchasingDocumentItem") == F.col("POA_Item")), how="left")
    # Enrich with companycode data product
    .join(company, F.col("CompanyCode") == F.col("CC_Key"), how="left")
    .drop("PO_Key", "POI_Key", "POI_Item", "POA_Key", "POA_Item", "CC_Key")
    # Apply authoritative sources with fallbacks:
    #   Supplier: purchaseorder (authoritative) > journalentry (fallback)
    #   GLAccount: purchaseorderaccountassignment (authoritative) > journalentry (fallback)
    # Enrich with supplierinvoice data product (authoritative Supplier per PO+POItem)
    .join(si_supplier,
          (F.col("PurchasingDocument") == F.col("SI_PO")) &
          (F.col("PurchasingDocumentItem") == F.col("SI_POItem")),
          how="left")
    .drop("SI_PO", "SI_POItem")
    #   Supplier: supplierinvoice (authoritative) > purchaseorder > operationalacctgdocitem
    .withColumn("Final_Supplier", F.coalesce(F.col("SI_Supplier"), F.col("PO_Supplier"), F.col("Supplier")))
    .withColumn("Final_GLAccount", F.coalesce(F.col("POA_GLAccount"), F.col("GLAccount")))
    # Compute signed amounts
    .withColumn("sign", F.when(F.col("DebitCreditCode") == "S", F.lit(1)).otherwise(F.lit(-1)))
    .withColumn("Signed_Amount", F.col("AmountInCompanyCodeCurrency") * F.col("sign"))
    .withColumn("Signed_Qty", F.col("PurchaseOrderQty") * F.col("sign"))
)

# ---- Aggregate GR at worklist key level ----
gr_agg = (
    gr_lines
    .groupBy(
        F.col("CompanyCode"),
        F.col("PurchasingDocument").alias("PurchaseOrder"),
        F.col("PurchasingDocumentItem").alias("PurchaseOrderItem"),
        F.col("Final_GLAccount").alias("GLAccount"),
        F.col("AccountingDocument"),
        F.col("Final_Supplier").alias("Supplier")
    )
    .agg(
        F.sum("Signed_Amount").alias("Signed_GR_Amount"),
        F.sum("Signed_Qty").alias("Signed_GR_Quantity"),
        F.min("PostingDate").alias("Oldest_GR_Date"),
        F.count("*").alias("GR_Line_Count"),
        # Enrichment fields from other data products
        F.first("PO_OrderQuantity").alias("PO_OrderQuantity"),
        F.first("PO_NetAmount").alias("PO_NetAmount"),
        F.first("PurchaseOrderItemText").alias("PurchaseOrderItemText"),
        F.first("DocumentCurrency").alias("DocumentCurrency"),
        F.first("CC_Currency").alias("CompanyCodeCurrency"),
        F.first("CC_CompanyCodeName").alias("CompanyCodeName"),
        # V2: Period awareness
        F.min("FiscalPeriod").alias("GR_FiscalPeriod"),
        F.min("FiscalYear").alias("GR_FiscalYear"),
        # V2: PO-level fields
        F.first("PO_Date").alias("PurchaseOrderDate"),
        F.first("PO_NetPrice").alias("PO_NetPrice"),
        F.first("PO_IsCompletelyDelivered").alias("IsCompletelyDelivered"),
        F.first("PO_IsFinallyInvoiced").alias("IsFinallyInvoiced")
    )
    .filter(F.col("Supplier").isNotNull())  # Vendor/Supplier = NOT NULL
)

gr_count = gr_agg.count()
gr_total = gr_agg.agg(F.sum('Signed_GR_Amount')).collect()[0][0]
print(f"GR Aggregation: {gr_count} PO/Item/Doc combinations with Goods Receipts")
print(f"Total Signed GR Amount: ${gr_total:,.2f}" if gr_total else "Total Signed GR Amount: $0.00 (no data - re-run synthetic data notebook)")
gr_agg.orderBy(F.col("Signed_GR_Amount").desc()).display()

GR Aggregation: 308 PO/Item/Doc combinations with Goods Receipts
Total Signed GR Amount: $94,160,117.82


CompanyCode,PurchaseOrder,PurchaseOrderItem,GLAccount,AccountingDocument,Supplier,Signed_GR_Amount,Signed_GR_Quantity,Oldest_GR_Date,GR_Line_Count,PO_OrderQuantity,PO_NetAmount,PurchaseOrderItemText,DocumentCurrency,CompanyCodeCurrency,CompanyCodeName,GR_FiscalPeriod,GR_FiscalYear,PurchaseOrderDate,PO_NetPrice,IsCompletelyDelivered,IsFinallyInvoiced
5010,4.500100059E9,20,51100000,5000000086,0050100019,4000000.0,0.0,2025-10-12,2,50.0,2000000.0,Control System Upgrade,CAD,EUR,Company Code EE 5010,10,2025,2025-08-28,40000.0,true,false
2210,4.500100043E9,10,62000000,5000000064,0050100028,4000000.0,0.0,2025-05-08,2,10.0,2000000.0,Refinery Turnaround Services,USD,CHF,Company Code CH 2210,5,2025,2025-04-25,200000.0,true,true
1010,4.500100052E9,10,61000000,5000000077,0050100004,2684153.56,0.0,2025-12-17,2,1.0,1342076.78,Cloud Migration Services,EUR,EUR,Company Code DE 1010,12,2025,2025-11-06,1342076.78,true,true
1010,4.500100044E9,10,66000000,5000000066,0050100010,2469510.62,0.0,2025-09-30,2,100.0,1234755.31,Electrical Switchgear Upgrade,EUR,EUR,Company Code DE 1010,9,2025,2025-08-28,12347.55,true,false
4010,4.500100156E9,20,64000000,5000000244,0050100023,2025583.12,0.0,2025-04-23,2,100.0,1012791.56,Tank Farm Maintenance,SGD,EUR,Company Code HR 4010,4,2025,2025-04-03,10127.92,true,false
1010,4.500100018E9,20,52000000,5000000030,0050100025,1889194.86,0.0,2025-03-31,2,5.0,944597.43,IT Infrastructure Upgrade,EUR,EUR,Company Code DE 1010,3,2025,2025-03-17,188919.49,true,false
2210,4.500100045E9,10,66000000,5000000067,0050100015,1741158.3,0.0,2025-09-04,2,20.0,870579.15,Cloud Migration Services,USD,CHF,Company Code CH 2210,9,2025,2025-07-26,43528.96,true,true
5010,4.500100097E9,10,65000000,5000000155,0050100011,1741146.78,0.0,2025-04-28,2,5.0,870573.39,Valve Replacement Program,CAD,EUR,Company Code EE 5010,4,2025,2025-03-27,174114.68,true,false
1010,4.500100079E9,20,63000000,5000000124,0050100003,1699623.88,0.0,2026-01-02,2,5.0,849811.94,Fire Protection System,EUR,EUR,Company Code DE 1010,1,2026,2025-12-14,169962.39,true,true
1010,4.50010006E9,10,62000000,5000000087,0050100030,1631551.98,0.0,2025-11-21,2,1000.0,815775.99,Polymer Additives Supply,EUR,EUR,Company Code DE 1010,11,2025,2025-10-27,815.78,true,false


In [0]:
# ── Step 2: IR (Invoice Receipt) — Multi-Data-Product Join ──────────────────
# Core: raw_journalentry (AccountingDocumentClass = 'RE', LedgerGroup = '0L')
# Same enrichment joins as GR for GLAccount authoritative source
# Signed IR Amount = SUM(AmountInCompanyCodeCurrency * sign(DebitCreditCode))

ir_lines = (
    je
    .filter(
        (F.col("AccountingDocumentClass") == "RE") &
        (F.col("PurchasingDocument").isNotNull()) &
        (F.col("DebitCreditCode").isin("S", "H"))
    )
    # Validate DebitCreditCode against reference table (journalentryitemcodes)
    .join(dcc, F.col("DebitCreditCode") == F.col("DCC_Code"), how="inner")
    .drop("DCC_Code")
    # Enrich with purchaseorderaccountassignment for authoritative GLAccount
    .join(po_acct, (F.col("PurchasingDocument") == F.col("POA_Key")) & 
                   (F.col("PurchasingDocumentItem") == F.col("POA_Item")), how="left")
    .drop("POA_Key", "POA_Item")
    # Enrich with purchaseorder for Supplier fallback
    .join(po_header, F.col("PurchasingDocument") == F.col("PO_Key"), how="left")
    .drop("PO_Key", "PO_CompanyCode", "DocumentCurrency", "PO_Date")
    # Supplier from supplierinvoice data product (authoritative per PO+POItem)
    .join(si_supplier,
          (F.col("PurchasingDocument") == F.col("SI_PO")) &
          (F.col("PurchasingDocumentItem") == F.col("SI_POItem")),
          how="left")
    .drop("SI_PO", "SI_POItem")
    .withColumn("Final_Supplier", F.coalesce(F.col("SI_Supplier"), F.col("PO_Supplier"), F.col("Supplier")))
    # Apply authoritative GLAccount
    .withColumn("Final_GLAccount", F.coalesce(F.col("POA_GLAccount"), F.col("GLAccount")))
    # Compute signed amounts
    .withColumn("sign", F.when(F.col("DebitCreditCode") == "S", F.lit(1)).otherwise(F.lit(-1)))
    .withColumn("Signed_Amount", F.col("AmountInCompanyCodeCurrency") * F.col("sign"))
    .withColumn("Signed_Qty", F.col("PurchaseOrderQty") * F.col("sign"))
)

# Aggregate IR at CompanyCode/PO/POItem/GLAccount level
# NOTE: AccountingDocument NOT in IR groupBy because IR docs differ from GR docs;
# the exposure join matches at PO/Item/GLAccount level.
ir_agg = (
    ir_lines
    .groupBy(
        F.col("CompanyCode"),
        F.col("PurchasingDocument").alias("PurchaseOrder"),
        F.col("PurchasingDocumentItem").alias("PurchaseOrderItem"),
        F.col("Final_GLAccount").alias("GLAccount")
    )
    .agg(
        F.sum("Signed_Amount").alias("Signed_IR_Amount"),
        F.sum("Signed_Qty").alias("Signed_IR_Quantity"),
        F.count("*").alias("IR_Line_Count"),
        # V2: Period awareness for cross-period check
        F.min("PostingDate").alias("Oldest_IR_Date"),
        F.min("FiscalPeriod").alias("IR_FiscalPeriod"),
        F.min("FiscalYear").alias("IR_FiscalYear")
    )
)

print(f"IR Aggregation: {ir_agg.count()} PO/Item combinations with Invoice Receipts")
print(f"Total Signed IR Amount: ${ir_agg.agg(F.sum('Signed_IR_Amount')).collect()[0][0]:,.2f}")
ir_agg.orderBy(F.col("Signed_IR_Amount").desc()).display()

IR Aggregation: 192 PO/Item combinations with Invoice Receipts
Total Signed IR Amount: $44,189,431.24


CompanyCode,PurchaseOrder,PurchaseOrderItem,GLAccount,Signed_IR_Amount,Signed_IR_Quantity,IR_Line_Count,Oldest_IR_Date,IR_FiscalPeriod,IR_FiscalYear
2210,4.500100043E9,10,62000000,4000000.0,0.0,2,2025-06-03,6,2025
1010,4.500100052E9,10,61000000,2684153.56,0.0,2,2025-12-29,12,2025
2210,4.500100045E9,10,66000000,1741158.3,0.0,2,2025-11-02,11,2025
1010,4.500100079E9,20,63000000,1699623.88,0.0,2,2026-02-02,2,2026
2210,4.500100006E9,30,62000000,1204760.4,0.0,2,2026-02-17,2,2026
5010,4.500100128E9,20,63000000,1169694.9,0.0,2,2026-02-14,2,2026
2210,4.500100041E9,10,66000000,1146898.24,0.0,2,2025-11-24,11,2025
5010,4.500100097E9,10,65000000,1044688.06,0.0,2,2025-06-06,6,2025
5010,4.500100128E9,10,61000000,739656.98,0.0,2,2026-01-30,1,2026
4010,4.500100156E9,20,64000000,729209.92,0.0,2,2025-06-18,6,2025


In [0]:
# ── Step 3: Compute Exposure & Days Open ─────────────────────────────────────
# Join GR to IR on CompanyCode/PO/POItem/GLAccount
# (NOT on AccountingDocument — GR and IR have different document numbers)
# Exposure = Signed GR Amount - Signed IR Amount (where GR > IR)
# Days Open = Current Date - Oldest GR posting date (doc type WE)
#
# NOTE (Multi-GR Allocation): If a single PO/Item/GLAccount has multiple GR
# documents, each GR row receives the FULL IR total in this join. For production
# data with partial deliveries, consider proportional allocation:
#   IR_per_GR = Total_IR * (GR_doc_amount / Total_GR_for_PO_item)
# In current synthetic data each PO item has exactly one GR doc, so no issue.

worklist_raw = (
    gr_agg
    .join(
        ir_agg,
        on=["CompanyCode", "PurchaseOrder", "PurchaseOrderItem", "GLAccount"],
        how="left"
    )
    .withColumn("Signed_IR_Amount", F.coalesce(F.col("Signed_IR_Amount"), F.lit(0.0)))
    .withColumn("Signed_IR_Quantity", F.coalesce(F.col("Signed_IR_Quantity"), F.lit(0.0)))
    .withColumn("IR_Line_Count", F.coalesce(F.col("IR_Line_Count"), F.lit(0)))
    .withColumn(
        "Exposure_Amount",
        F.when(
            F.col("Signed_GR_Amount") > F.col("Signed_IR_Amount"),
            F.col("Signed_GR_Amount") - F.col("Signed_IR_Amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "Days_Open",
        F.datediff(
            F.current_date(),
            F.to_date(F.col("Oldest_GR_Date"), "yyyy-MM-dd")
        )
    )
    # Convert Exposure to USD using static rates
    # (Replace with live FX rate table in production)
    .withColumn(
        "USD_Rate",
        F.when(F.col("CompanyCodeCurrency") == "USD", F.lit(1.0))
        .when(F.col("CompanyCodeCurrency") == "EUR", F.lit(1.08))
        .when(F.col("CompanyCodeCurrency") == "GBP", F.lit(1.27))
        .when(F.col("CompanyCodeCurrency") == "CHF", F.lit(1.12))
        .when(F.col("CompanyCodeCurrency") == "CAD", F.lit(0.74))
        .when(F.col("CompanyCodeCurrency") == "SGD", F.lit(0.75))
        .when(F.col("CompanyCodeCurrency") == "AUD", F.lit(0.66))
        .when(F.col("CompanyCodeCurrency") == "JPY", F.lit(0.0064))
        .when(F.col("CompanyCodeCurrency") == "INR", F.lit(0.012))
        .when(F.col("CompanyCodeCurrency") == "CNY", F.lit(0.14))
        .when(F.col("CompanyCodeCurrency") == "BRL", F.lit(0.18))
        .otherwise(F.lit(1.0))  # Default to 1:1 for unmapped currencies
    )
    .withColumn("Exposure_Amount_USD", F.round(F.col("Exposure_Amount") * F.col("USD_Rate"), 2))
    .drop("USD_Rate")
    # Fiscal Year and Period derived from Oldest GR PostingDate
    .withColumn("Fiscal_Year", F.year(F.to_date(F.col("Oldest_GR_Date"), "yyyy-MM-dd")))
    .withColumn("Fiscal_Period", F.month(F.to_date(F.col("Oldest_GR_Date"), "yyyy-MM-dd")))
    # Filter: only items with positive exposure (GR > IR)
    .filter(F.col("Exposure_Amount") > 0)
)

print(f"Worklist (with exposure > 0): {worklist_raw.count()} items")
print(f"Total Exposure: ${worklist_raw.agg(F.sum('Exposure_Amount')).collect()[0][0]:,.2f}")
worklist_raw.display()

Worklist (with exposure > 0): 203 items
Total Exposure: $50,060,493.62


CompanyCode,PurchaseOrder,PurchaseOrderItem,GLAccount,AccountingDocument,Supplier,Signed_GR_Amount,Signed_GR_Quantity,Oldest_GR_Date,GR_Line_Count,PO_OrderQuantity,PO_NetAmount,PurchaseOrderItemText,DocumentCurrency,CompanyCodeCurrency,CompanyCodeName,GR_FiscalPeriod,GR_FiscalYear,PurchaseOrderDate,PO_NetPrice,IsCompletelyDelivered,IsFinallyInvoiced,Signed_IR_Amount,Signed_IR_Quantity,IR_Line_Count,Oldest_IR_Date,IR_FiscalPeriod,IR_FiscalYear,Exposure_Amount,Days_Open,Exposure_Amount_USD,Fiscal_Year,Fiscal_Period
2210,4.500100011E9,20,51100000,5000000022,0050100017,178780.88,0.0,2025-04-10,2,5.0,89390.44,Consulting - Digital Twin,USD,CHF,Company Code CH 2210,4,2025,2025-03-28,17878.09,true,false,0.0,0.0,0,null,null,null,178780.88,428,200234.59,2025,4
3010,4.50010002E9,10,63000000,5000000033,0050100018,88361.68,0.0,2025-08-07,2,50.0,44180.84,Boiler Tube Inspection,GBP,AUD,Company Code AU 3010,8,2025,2025-06-29,883.62,true,false,0.0,0.0,0,null,null,null,88361.68,309,58318.71,2025,8
3010,4.500100021E9,10,63000000,5000000034,0050100007,108773.92,0.0,2025-12-27,2,10.0,54386.96,Refinery Turnaround Services,GBP,AUD,Company Code AU 3010,12,2025,2025-12-06,5438.7,true,false,32632.18,0.0,2,2026-02-26,2,2026,76141.73999999999,167,50253.55,2025,12
2210,4.500100033E9,20,62000000,5000000051,0050100025,185172.32,0.0,2025-07-27,2,5.0,92586.16,Lab Equipment Maintenance,USD,CHF,Company Code CH 2210,7,2025,2025-07-15,18517.23,true,false,0.0,0.0,0,null,null,null,185172.32,320,207393.0,2025,7
4010,4.500100036E9,20,63000000,5000000055,0050100030,104697.24,0.0,2025-06-09,2,20.0,52348.62,SAP S/4HANA License Renewal,SGD,EUR,Company Code HR 4010,6,2025,2025-04-30,2617.43,true,false,0.0,0.0,0,null,null,null,104697.24,368,113073.02,2025,6
2210,4.500100065E9,10,65000000,5000000094,0050100001,1117480.96,0.0,2026-03-04,2,50.0,558740.48,Rotating Equipment Repair,USD,CHF,Company Code CH 2210,3,2026,2026-02-02,11174.81,true,false,0.0,0.0,0,null,null,null,1117480.96,100,1251578.68,2026,3
2210,4.500100066E9,20,63000000,5000000097,0050100003,226630.38,0.0,2025-03-28,2,100.0,113315.19,Compressor Overhaul,USD,CHF,Company Code CH 2210,3,2025,2025-03-14,1133.15,true,false,0.0,0.0,0,null,null,null,226630.38,441,253826.03,2025,3
4010,4.500100113E9,20,61000000,5000000180,0050100017,49329.06,0.0,2025-09-29,2,1000.0,24664.53,Project Management Services,SGD,EUR,Company Code HR 4010,9,2025,2025-09-06,24.66,true,false,0.0,0.0,0,null,null,null,49329.06,256,53275.38,2025,9
2210,4.500100122E9,10,62000000,5000000190,0050100025,67600.98,0.0,2025-08-03,2,50.0,33800.49,IT Infrastructure Upgrade,USD,CHF,Company Code CH 2210,8,2025,2025-07-10,676.01,true,false,0.0,0.0,0,null,null,null,67600.98,313,75713.1,2025,8
1010,4.500100135E9,10,66000000,5000000213,0050100021,809786.7,0.0,2026-01-18,2,50.0,404893.35,Lab Equipment Maintenance,EUR,EUR,Company Code DE 1010,1,2026,2026-01-06,8097.87,true,false,0.0,0.0,0,null,null,null,809786.7,145,874569.64,2026,1


In [0]:
# ── Step 3B: V2 — Variance Classification & Worklist Category ─────────────
# Extension of V1. All V1 measures reused as-is.
# V2 adds: period awareness, variance classification, payment netting, worklist category.

# ---- V2 Computations on top of worklist_raw ----
worklist_v2 = (
    worklist_raw
    # --- 1. Posting Period Alignment (Cut-off Check) ---
    # Cross-Period Flag: GR posted in earlier period than IR
    .withColumn(
        "GR_Period_Key",
        F.concat_ws("-", F.col("GR_FiscalYear").cast("string"), 
                    F.lpad(F.col("GR_FiscalPeriod").cast("string"), 2, "0"))
    )
    .withColumn(
        "IR_Period_Key",
        F.concat_ws("-", F.col("IR_FiscalYear").cast("string"), 
                    F.lpad(F.col("IR_FiscalPeriod").cast("string"), 2, "0"))
    )
    .withColumn(
        "Cross_Period_Flag",
        F.when(
            (F.col("IR_FiscalYear").isNotNull()) &
            (F.col("GR_Period_Key") < F.col("IR_Period_Key")),
            F.lit("Y")
        ).otherwise(F.lit("N"))
    )
    # --- 2. Quantity Variance ---
    .withColumn("Quantity_Variance", F.col("Signed_GR_Quantity") - F.col("Signed_IR_Quantity"))
    .withColumn(
        "Net_Order_Price",
        F.when(
            (F.col("PO_NetPrice").isNotNull()) & (F.col("PO_NetPrice") != 0),
            F.col("PO_NetPrice")
        ).otherwise(
            # Fallback: derive from PO_NetAmount / PO_OrderQuantity
            F.when(
                (F.col("PO_OrderQuantity").isNotNull()) & (F.col("PO_OrderQuantity") != 0),
                F.col("PO_NetAmount") / F.col("PO_OrderQuantity")
            )
        )
    )
    .withColumn(
        "Quantity_Exposure",
        F.when(F.col("Quantity_Variance") > 0,
               F.round(F.col("Quantity_Variance") * F.col("Net_Order_Price"), 2)
        ).otherwise(F.lit(0.0))
    )
    # --- 3. Price / Billing Variance ---
    # Condition: Quantities match but amounts differ
    .withColumn(
        "Billing_Variance",
        F.when(
            (F.abs(F.col("Quantity_Variance")) < 0.01) &  # Qty matched (within tolerance)
            (F.abs(F.col("Signed_GR_Amount") - F.col("Signed_IR_Amount")) > 0.01),
            F.col("Signed_GR_Amount") - F.col("Signed_IR_Amount")
        ).otherwise(F.lit(0.0))
    )
    # --- 4. Missing Invoice Flag ---
    # GR exists, no IR at all (Signed_IR_Amount = 0 AND IR_Line_Count = 0)
    .withColumn(
        "Invoice_Missing_Flag",
        F.when(
            (F.col("Signed_GR_Amount") > 0) & (F.col("IR_Line_Count") == 0),
            F.lit("Y")
        ).otherwise(F.lit("N"))
    )
    # --- 5. Days Open refinement (overwrite with PO Date fallback) ---
    .withColumn(
        "Days_Open",
        F.coalesce(
            F.col("Days_Open"),
            F.datediff(F.current_date(), F.to_date(F.col("PurchaseOrderDate"), "yyyy-MM-dd"))
        )
    )
    # --- 6. Open Amount Net of Payments (simplified) ---
    # Down payments: not yet available in data - placeholder for future
    # Paid Amount = 0 (conservative - overstated, never understated)
    .withColumn("Paid_Amount", F.lit(0.0))  # TODO: integrate clearing/payment when available
    .withColumn("GR_IR_Accrual", F.col("Exposure_Amount"))  # Same as V1 Exposure
    .withColumn("Open_Payable",
        F.when(F.col("Signed_IR_Amount") > F.col("Paid_Amount"),
               F.col("Signed_IR_Amount") - F.col("Paid_Amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn("Open_Amount_Against_PO",
        F.col("GR_IR_Accrual") + F.col("Open_Payable")
    )
    # --- 7. Worklist Category (first match wins, top-down) ---
    .withColumn(
        "Worklist_Category",
        F.when(
            # 1. Invoice Missing — GR exists, no RE posting at all
            F.col("Invoice_Missing_Flag") == "Y",
            F.lit("Invoice Missing")
        ).when(
            # 2. Cross-Period GR-IR — GR period earlier than IR period
            F.col("Cross_Period_Flag") == "Y",
            F.lit("Cross-Period GR-IR")
        ).when(
            # 3. Quantity Variance — GR Qty ≠ IR Qty
            F.abs(F.col("Quantity_Variance")) >= 0.01,
            F.when(F.col("Quantity_Variance") > 0, F.lit("Quantity Variance / Partial GR-IR"))
             .otherwise(F.lit("Over-Billing (IR > GR Qty)"))
        ).when(
            # 4. Price Variance — quantity matched, amount differs
            F.abs(F.col("Billing_Variance")) > 0.01,
            F.lit("Price Variance")
        ).when(
            # 5. Matched – Unpaid — GR ≈ IR but Open Payable > 0
            (F.col("Exposure_Amount") < 0.01) & (F.col("Open_Payable") > 0),
            F.lit("Matched - Unpaid")
        ).otherwise(
            # 6. Fully Matched & Paid — should not appear (filtered by Exposure > 0)
            F.lit("Unclassified")
        )
    )
    # --- Exclusion: skip items where delivery/invoice is marked complete ---
    .withColumn(
        "Exclude_From_Worklist",
        F.when(
            (F.col("Invoice_Missing_Flag") == "Y") &
            (
                (F.col("IsFinallyInvoiced") == True) |
                (F.col("IsCompletelyDelivered") == True)
            ),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .filter(F.col("Exclude_From_Worklist") == False)
    .drop("Exclude_From_Worklist")
)

print(f"V2 Worklist: {worklist_v2.count()} items")
print(f"\nWorklist Category Distribution:")
worklist_v2.groupBy("Worklist_Category").agg(
    F.count("*").alias("Items"),
    F.round(F.sum("Exposure_Amount"), 2).alias("Total_Exposure"),
    F.round(F.sum("Exposure_Amount_USD"), 2).alias("Total_Exposure_USD")
).orderBy("Worklist_Category").show(truncate=False)

V2 Worklist: 87 items

Worklist Category Distribution:
+------------------+-----+--------------+------------------+
|Worklist_Category |Items|Total_Exposure|Total_Exposure_USD|
+------------------+-----+--------------+------------------+
|Cross-Period GR-IR|78   |1.196567136E7 |1.2145706E7       |
|Price Variance    |9    |785058.1      |816264.67         |
+------------------+-----+--------------+------------------+



In [0]:
# ── Step 4: Assign Recommended Finance Action ────────────────────────────────
# Deterministic rule-based logic:
#   POST:       Exposure > $50,000 AND Days Open > 30 (high value, overdue)
#   REVIEW:     Exposure > $20,000 OR Days Open > 45 (needs attention)
#   OBSERVE:    Exposure > $5,000 AND Days Open between 15-30 (watch list)
#   NO ACTION:  Everything else (low risk)

worklist_final = (
    worklist_v2
    .withColumn(
        "Recommended_Action",
        F.when(
            (F.col("Exposure_Amount") > 50000) & (F.col("Days_Open") > 30),
            F.lit("POST")
        ).when(
            (F.col("Exposure_Amount") > 20000) | (F.col("Days_Open") > 45),
            F.lit("REVIEW")
        ).when(
            (F.col("Exposure_Amount") > 5000) & (F.col("Days_Open").between(15, 30)),
            F.lit("OBSERVE")
        ).otherwise(
            F.lit("NO ACTION")
        )
    )
    # ── RCA (Root Cause Analysis) — deterministic, data-driven narrative per category ──
    .withColumn(
        "RCA",
        F.when(
            # 1. Invoice Missing — GR exists, no RE posting (Scenario 4)
            F.col("Worklist_Category") == "Invoice Missing",
            F.concat(
                F.lit("Invoice Missing: Goods receipt posted (GR amount "),
                F.round(F.col("Signed_GR_Amount"), 2).cast("string"),
                F.lit(" "), F.col("CompanyCodeCurrency"),
                F.lit(") but no corresponding invoice receipt (RE) exists. "),
                F.lit("Open for "), F.col("Days_Open").cast("string"), F.lit(" days. "),
                F.lit("Likely cause: vendor invoice delayed, lost, or not yet submitted.")
            )
        ).when(
            # 2. Cross-Period GR-IR — cut-off accrual (Scenario 1)
            F.col("Worklist_Category") == "Cross-Period GR-IR",
            F.concat(
                F.lit("Cross-Period Cut-Off: GR posted in period "),
                F.col("GR_FiscalYear").cast("string"), F.lit("/"),
                F.lpad(F.col("GR_FiscalPeriod").cast("string"), 2, "0"),
                F.lit(" but IR posted in later period "),
                F.col("IR_FiscalYear").cast("string"), F.lit("/"),
                F.lpad(F.col("IR_FiscalPeriod").cast("string"), 2, "0"),
                F.lit(". Exposure "), F.round(F.col("Exposure_Amount"), 2).cast("string"),
                F.lit(" "), F.col("CompanyCodeCurrency"),
                F.lit(" requires period-end accrual adjustment.")
            )
        ).when(
            # 3. Quantity Variance / Partial GR-IR (Scenario 2)
            F.col("Worklist_Category") == "Quantity Variance / Partial GR-IR",
            F.concat(
                F.lit("Quantity Variance: GR qty ("),
                F.round(F.col("Signed_GR_Quantity"), 2).cast("string"),
                F.lit(") exceeds IR qty ("),
                F.round(F.col("Signed_IR_Quantity"), 2).cast("string"),
                F.lit("). Unmatched qty: "),
                F.round(F.col("Quantity_Variance"), 2).cast("string"),
                F.lit(" units @ "),
                F.round(F.col("Net_Order_Price"), 2).cast("string"),
                F.lit(" = exposure "),
                F.round(F.col("Quantity_Exposure"), 2).cast("string"),
                F.lit(" "), F.col("CompanyCodeCurrency"),
                F.lit(". Likely cause: partial delivery invoiced or GR over-receipt.")
            )
        ).when(
            # 3b. Over-Billing (IR > GR Qty)
            F.col("Worklist_Category") == "Over-Billing (IR > GR Qty)",
            F.concat(
                F.lit("Over-Billing: IR qty ("),
                F.round(F.col("Signed_IR_Quantity"), 2).cast("string"),
                F.lit(") exceeds GR qty ("),
                F.round(F.col("Signed_GR_Quantity"), 2).cast("string"),
                F.lit("). Vendor may have invoiced more than received. Review for credit memo or return.")
            )
        ).when(
            # 4. Price Variance (Scenario 3)
            F.col("Worklist_Category") == "Price Variance",
            F.concat(
                F.lit("Price Variance: Quantities match but billing variance of "),
                F.round(F.col("Billing_Variance"), 2).cast("string"),
                F.lit(" "), F.col("CompanyCodeCurrency"),
                F.lit(" detected (GR amount "),
                F.round(F.col("Signed_GR_Amount"), 2).cast("string"),
                F.lit(" vs IR amount "),
                F.round(F.col("Signed_IR_Amount"), 2).cast("string"),
                F.lit("). Likely cause: price change, surcharge, or invoicing error vs PO terms.")
            )
        ).when(
            # 5. Matched – Unpaid (Scenario 6)
            F.col("Worklist_Category") == "Matched - Unpaid",
            F.concat(
                F.lit("Matched-Unpaid: GR and IR amounts are matched (exposure ~0) but open payable of "),
                F.round(F.col("Open_Payable"), 2).cast("string"),
                F.lit(" "), F.col("CompanyCodeCurrency"),
                F.lit(" remains outstanding. Payment not yet cleared against vendor account.")
            )
        ).otherwise(
            F.lit("Unclassified: Item does not match standard variance patterns. Manual review recommended.")
        )
    )
    .select(
        # Worklist key (from multiple data products)
        "CompanyCode",
        "PurchaseOrder",
        "PurchaseOrderItem",
        "GLAccount",
        "AccountingDocument",
        "Supplier",
        # Enrichment from purchaseorder data product
        "PurchaseOrderItemText",
        "DocumentCurrency",
        "PO_OrderQuantity",
        "PO_NetAmount",
        # Enrichment from companycode data product
        "CompanyCodeCurrency",
        "CompanyCodeName",
        # V1 Computed fields
        "Oldest_GR_Date",
        "Signed_GR_Amount",
        "Signed_GR_Quantity",
        "Signed_IR_Amount",
        "Signed_IR_Quantity",
        "GR_Line_Count",
        "IR_Line_Count",
        "Exposure_Amount",
        "Exposure_Amount_USD",
        "Days_Open",
        "Recommended_Action",
        # V2: Period Awareness
        "GR_FiscalYear",
        "GR_FiscalPeriod",
        "IR_FiscalYear",
        "IR_FiscalPeriod",
        "Cross_Period_Flag",
        # V2: Variance Classification
        "Quantity_Variance",
        "Quantity_Exposure",
        "Net_Order_Price",
        "Billing_Variance",
        "Invoice_Missing_Flag",
        # V2: Payment Netting
        "GR_IR_Accrual",
        "Open_Payable",
        "Open_Amount_Against_PO",
        # PO Date (used as Days Open fallback)
        "PurchaseOrderDate",
        # Fiscal Year/Period from Oldest GR PostingDate
        "Fiscal_Year",
        "Fiscal_Period",
        # V2: Worklist Category
        "Worklist_Category",
        # V2: Root Cause Analysis
        "RCA"
    )
    .orderBy(F.col("Exposure_Amount").desc())
)

print(f"\n{'=' * 70}")
print(f"  ACCRUAL WORKLIST V2 SUMMARY")
print(f"{'=' * 70}")
total_items = worklist_final.count()
total_exp = worklist_final.agg(F.sum('Exposure_Amount')).collect()[0][0]
total_exp_usd = worklist_final.agg(F.sum('Exposure_Amount_USD')).collect()[0][0]
print(f"  Total Items:          {total_items}")
print(f"  Total Exposure (LC):  ${total_exp:,.2f}" if total_exp else "  Total Exposure: $0.00")
print(f"  Total Exposure (USD): ${total_exp_usd:,.2f}" if total_exp_usd else "  Total Exposure USD: $0.00")
print(f"{'-' * 70}")
print(f"  Worklist Category Breakdown:")
worklist_final.groupBy("Worklist_Category").agg(
    F.count("*").alias("Items"),
    F.round(F.sum("Exposure_Amount"), 2).alias("Exposure_LC"),
    F.round(F.sum("Exposure_Amount_USD"), 2).alias("Exposure_USD")
).orderBy(F.desc("Exposure_USD")).show(truncate=False)
print(f"{'-' * 70}")
print(f"  Recommended Action Breakdown:")
worklist_final.groupBy("Recommended_Action").agg(
    F.count("*").alias("Items"),
    F.round(F.sum("Exposure_Amount_USD"), 2).alias("Exposure_USD")
).orderBy("Recommended_Action").show()

worklist_final.display()


  ACCRUAL WORKLIST V2 SUMMARY
  Total Items:          87
  Total Exposure (LC):  $12,750,729.46
  Total Exposure (USD): $12,961,970.67
----------------------------------------------------------------------
  Worklist Category Breakdown:
+------------------+-----+-------------+------------+
|Worklist_Category |Items|Exposure_LC  |Exposure_USD|
+------------------+-----+-------------+------------+
|Cross-Period GR-IR|78   |1.196567136E7|1.2145706E7 |
|Price Variance    |9    |785058.1     |816264.67   |
+------------------+-----+-------------+------------+

----------------------------------------------------------------------
  Recommended Action Breakdown:
+------------------+-----+-------------+
|Recommended_Action|Items| Exposure_USD|
+------------------+-----+-------------+
|              POST|   56|1.219143158E7|
|            REVIEW|   31|    770539.09|
+------------------+-----+-------------+



CompanyCode,PurchaseOrder,PurchaseOrderItem,GLAccount,AccountingDocument,Supplier,PurchaseOrderItemText,DocumentCurrency,PO_OrderQuantity,PO_NetAmount,CompanyCodeCurrency,CompanyCodeName,Oldest_GR_Date,Signed_GR_Amount,Signed_GR_Quantity,Signed_IR_Amount,Signed_IR_Quantity,GR_Line_Count,IR_Line_Count,Exposure_Amount,Exposure_Amount_USD,Days_Open,Recommended_Action,GR_FiscalYear,GR_FiscalPeriod,IR_FiscalYear,IR_FiscalPeriod,Cross_Period_Flag,Quantity_Variance,Quantity_Exposure,Net_Order_Price,Billing_Variance,Invoice_Missing_Flag,GR_IR_Accrual,Open_Payable,Open_Amount_Against_PO,PurchaseOrderDate,Fiscal_Year,Fiscal_Period,Worklist_Category,RCA
4010,4.500100156E9,20,64000000,5000000244,0050100023,Tank Farm Maintenance,SGD,100.0,1012791.56,EUR,Company Code HR 4010,2025-04-23,2025583.12,0.0,729209.92,0.0,2,2,1296373.2000000002,1400083.06,415,POST,2025,4,2025,6,Y,0.0,0.0,10127.92,1296373.2000000002,N,1296373.2000000002,729209.92,2025583.12,2025-04-03,2025,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/04 but IR posted in later period 2025/06. Exposure 1296373.2 EUR requires period-end accrual adjustment.
2210,4.500100051E9,30,52000000,5000000076,0050100029,Control System Upgrade,USD,20.0,369390.69,CHF,Company Code CH 2210,2026-04-02,738781.38,0.0,36939.06,0.0,2,2,701842.3200000001,786063.4,71,POST,2026,4,2026,5,Y,0.0,0.0,18469.53,701842.3200000001,N,701842.3200000001,36939.06,738781.3800000001,2026-02-22,2026,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2026/04 but IR posted in later period 2026/05. Exposure 701842.32 CHF requires period-end accrual adjustment.
5010,4.500100097E9,10,65000000,5000000155,0050100011,Valve Replacement Program,CAD,5.0,870573.39,EUR,Company Code EE 5010,2025-04-28,1741146.78,0.0,1044688.06,0.0,2,2,696458.72,752175.42,410,POST,2025,4,2025,6,Y,0.0,0.0,174114.68,696458.72,N,696458.72,1044688.06,1741146.78,2025-03-27,2025,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/04 but IR posted in later period 2025/06. Exposure 696458.72 EUR requires period-end accrual adjustment.
1010,4.500100148E9,30,66000000,5000000233,0050100006,Project Management Services,EUR,20.0,451199.42,EUR,Company Code DE 1010,2025-07-26,902398.84,0.0,270719.66,0.0,2,2,631679.1799999999,682213.51,321,POST,2025,7,2025,9,Y,0.0,0.0,22559.97,631679.1799999999,N,631679.1799999999,270719.66,902398.8399999999,2025-07-13,2025,7,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/07 but IR posted in later period 2025/09. Exposure 631679.18 EUR requires period-end accrual adjustment.
1010,4.500100086E9,10,62000000,5000000139,0050100005,Control System Upgrade,EUR,50.0,387209.09,EUR,Company Code DE 1010,2025-12-07,774418.18,0.0,201348.72,0.0,2,2,573069.4600000001,618915.02,187,POST,2025,12,2026,1,Y,0.0,0.0,7744.18,573069.4600000001,N,573069.4600000001,201348.72,774418.18,2025-10-25,2025,12,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/12 but IR posted in later period 2026/01. Exposure 573069.46 EUR requires period-end accrual adjustment.
3010,4.500100132E9,20,63000000,5000000210,0050100005,Catalytic Cracker Maintenance,GBP,20.0,266107.95,AUD,Company Code AU 3010,2026-04-02,532215.9,0.0,26610.8,0.0,2,2,505605.10000000003,333699.37,71,POST,2026,4,2026,6,Y,0.0,0.0,13305.4,505605.10000000003,N,505605.10000000003,26610.8,532215.9,2026-03-12,2026,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2026/04 but IR posted in later period 2026/06. Exposure 505605.1 AUD requires period-end accrual adjustment.
5010,4.500100101E9,20,63000000,5000000162,0050100021,Pipeline Integrity Assessment,CAD,10.0,324516.14,EUR,Company Code EE 5010,2026-01-02,649032.28,0.0,194709.68,0.0,2,2,454322.60000000003,490668.41,161,POST,2026,1,2026,2,Y,0.0,0.0,32451.61,454322.60000000003,N,454322.60000000003,194709.68,649032.28,2025-12-01,2026,1,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2026/01 but IR posted in later period 2026/02. Exposure 454322.6 EUR requires pe

In [0]:
# ── Step 5: Persist as Silver Delta Table ────────────────────────────────────
# Round amount columns to 2 decimal places for clean storage
amount_cols = [
    "Signed_GR_Amount", "Signed_IR_Amount",
    "Exposure_Amount", "Exposure_Amount_USD",
    "Quantity_Exposure", "Net_Order_Price",
    "Billing_Variance", "GR_IR_Accrual",
    "Open_Payable", "Open_Amount_Against_PO",
    "PO_NetAmount"
]
worklist_rounded = worklist_final
for col_name in amount_cols:
    worklist_rounded = worklist_rounded.withColumn(col_name, F.round(F.col(col_name), 2))

worklist_rounded.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE)

print(f"✓ Saved to: {CATALOG}.{SCHEMA_SILVER}.{TARGET_TABLE}")
print(f"  Rows written: {spark.table(TARGET_TABLE).count()}")
print(f"  Amount columns rounded to 2 decimals: {len(amount_cols)} columns")

# Display final table
spark.table(TARGET_TABLE).display()

✓ Saved to: [pe]_xx102_pe_finance.silver.accrual_worklist
  Rows written: 87
  Amount columns rounded to 2 decimals: 11 columns


CompanyCode,PurchaseOrder,PurchaseOrderItem,GLAccount,AccountingDocument,Supplier,PurchaseOrderItemText,DocumentCurrency,PO_OrderQuantity,PO_NetAmount,CompanyCodeCurrency,CompanyCodeName,Oldest_GR_Date,Signed_GR_Amount,Signed_GR_Quantity,Signed_IR_Amount,Signed_IR_Quantity,GR_Line_Count,IR_Line_Count,Exposure_Amount,Exposure_Amount_USD,Days_Open,Recommended_Action,GR_FiscalYear,GR_FiscalPeriod,IR_FiscalYear,IR_FiscalPeriod,Cross_Period_Flag,Quantity_Variance,Quantity_Exposure,Net_Order_Price,Billing_Variance,Invoice_Missing_Flag,GR_IR_Accrual,Open_Payable,Open_Amount_Against_PO,PurchaseOrderDate,Fiscal_Year,Fiscal_Period,Worklist_Category,RCA
4010,4.500100156E9,20,64000000,5000000244,0050100023,Tank Farm Maintenance,SGD,100.0,1012791.56,EUR,Company Code HR 4010,2025-04-23,2025583.12,0.0,729209.92,0.0,2,2,1296373.2,1400083.06,415,POST,2025,4,2025,6,Y,0.0,0.0,10127.92,1296373.2,N,1296373.2,729209.92,2025583.12,2025-04-03,2025,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/04 but IR posted in later period 2025/06. Exposure 1296373.2 EUR requires period-end accrual adjustment.
2210,4.500100051E9,30,52000000,5000000076,0050100029,Control System Upgrade,USD,20.0,369390.69,CHF,Company Code CH 2210,2026-04-02,738781.38,0.0,36939.06,0.0,2,2,701842.32,786063.4,71,POST,2026,4,2026,5,Y,0.0,0.0,18469.53,701842.32,N,701842.32,36939.06,738781.38,2026-02-22,2026,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2026/04 but IR posted in later period 2026/05. Exposure 701842.32 CHF requires period-end accrual adjustment.
5010,4.500100097E9,10,65000000,5000000155,0050100011,Valve Replacement Program,CAD,5.0,870573.39,EUR,Company Code EE 5010,2025-04-28,1741146.78,0.0,1044688.06,0.0,2,2,696458.72,752175.42,410,POST,2025,4,2025,6,Y,0.0,0.0,174114.68,696458.72,N,696458.72,1044688.06,1741146.78,2025-03-27,2025,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/04 but IR posted in later period 2025/06. Exposure 696458.72 EUR requires period-end accrual adjustment.
1010,4.500100148E9,30,66000000,5000000233,0050100006,Project Management Services,EUR,20.0,451199.42,EUR,Company Code DE 1010,2025-07-26,902398.84,0.0,270719.66,0.0,2,2,631679.18,682213.51,321,POST,2025,7,2025,9,Y,0.0,0.0,22559.97,631679.18,N,631679.18,270719.66,902398.84,2025-07-13,2025,7,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/07 but IR posted in later period 2025/09. Exposure 631679.18 EUR requires period-end accrual adjustment.
1010,4.500100086E9,10,62000000,5000000139,0050100005,Control System Upgrade,EUR,50.0,387209.09,EUR,Company Code DE 1010,2025-12-07,774418.18,0.0,201348.72,0.0,2,2,573069.46,618915.02,187,POST,2025,12,2026,1,Y,0.0,0.0,7744.18,573069.46,N,573069.46,201348.72,774418.18,2025-10-25,2025,12,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/12 but IR posted in later period 2026/01. Exposure 573069.46 EUR requires period-end accrual adjustment.
3010,4.500100132E9,20,63000000,5000000210,0050100005,Catalytic Cracker Maintenance,GBP,20.0,266107.95,AUD,Company Code AU 3010,2026-04-02,532215.9,0.0,26610.8,0.0,2,2,505605.1,333699.37,71,POST,2026,4,2026,6,Y,0.0,0.0,13305.4,505605.1,N,505605.1,26610.8,532215.9,2026-03-12,2026,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2026/04 but IR posted in later period 2026/06. Exposure 505605.1 AUD requires period-end accrual adjustment.
5010,4.500100101E9,20,63000000,5000000162,0050100021,Pipeline Integrity Assessment,CAD,10.0,324516.14,EUR,Company Code EE 5010,2026-01-02,649032.28,0.0,194709.68,0.0,2,2,454322.6,490668.41,161,POST,2026,1,2026,2,Y,0.0,0.0,32451.61,454322.6,N,454322.6,194709.68,649032.28,2025-12-01,2026,1,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2026/01 but IR posted in later period 2026/02. Exposure 454322.6 EUR requires period-end accrual adjustment.
3010,4.500100129E9,20,64000000,5000000204,0050100015,Polymer Additives Supply,GBP,500.0,336075.17,AUD,Company Code AU 3010,2025-07-09,672150.34,0.